## STATS 507: Data Science and Analytics using Python
### Final Project: Fine-Tuning DistilBERT for Sentiment Analysis on Movie Review Datasets: A Comparative Study
#### Programmer: Liangyue Zhou larryzh@umich.edu

##### In this project, we aim to construct a sentimental analysis model for classifying movie reviews. We firstly try out a simple logistic regression model. We then try to fine-tune a DistilBERT model.

#### Step 1. Install and import necessary libraries and load datasets

In [2]:
!pip install -q datasets evaluate

In [4]:
from datasets import load_dataset, concatenate_datasets

dataset_rt = load_dataset("cornell-movie-review-data/rotten_tomatoes")
dataset_imdb = load_dataset("jahjinx/IMDb_movie_reviews")

In [5]:
import random
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
def sample_from_dataset(dataset,split = 'train',sample_size=5):

    df = dataset[split].to_pandas()
    sampled_df = df.sample(n=sample_size, random_state=34)

    return sampled_df

# Sample data from each dataset
print(sample_from_dataset(dataset_rt))
print(sample_from_dataset(dataset_imdb))

                                                   text  label
687   this is a raw and disturbing tale that took fi...      1
7373  i guess it just goes to show that if you give ...      0
1114  this surreal gilliam-esque film is also a trou...      1
7135  i wish i could say " thank god it's friday " ,...      0
8506  the film desperately sinks further and further...      0
                                                    text  label
18890  This solid black and white slapstick comedy wi...      1
8642   This wonderful movie captures so many elements...      1
14796  I was really surprised with this movie. Going ...      1
31427  I have NEVER EVER seen such a bad movie before...      0
34195  This is a made-for-TV and rather needless Sci-...      0


#### Step 2. Use simple vectorization and logistic regression to classify movie reviews. First, we train the model using the IMDb movie reviews training dataset and evaluate its performance on the corresponding test dataset. Next, we assess the model's performance on the test dataset from Rotten Tomatoes movie reviews.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [8]:
df_train = dataset_imdb['train'].to_pandas()
texts_train = df_train['text']
labels_train = df_train['label']

vectorizer = CountVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(texts_train)

classifier = LogisticRegression(max_iter=200,C=0.05)
classifier.fit(X_train, labels_train)

df_test = dataset_imdb['test'].to_pandas()
texts_test = df_test['text']
labels_test = df_test['label']

X_test = vectorizer.transform(texts_test)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(labels_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(classification_report(labels_test, y_pred))

Accuracy: 0.89

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      5044
           1       0.88      0.90      0.89      4956

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [9]:
df_test = dataset_rt['test'].to_pandas()
texts_test = df_test['text']
labels_test = df_test['label']

X_test = vectorizer.transform(texts_test)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(labels_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(classification_report(labels_test, y_pred))

Accuracy: 0.74

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.65      0.71       533
           1       0.70      0.82      0.76       533

    accuracy                           0.74      1066
   macro avg       0.74      0.74      0.73      1066
weighted avg       0.74      0.74      0.73      1066



**Comment**: The resulting accuracy appears satisfactory on the IMDb movie reviews test dataset, but when the test set is switched to reviews from the rotten tomatoes dataset, the accuracy drops significantly. This decline could be attributed to differences in writing styles between the two datasets. Reviews from rotten tomatoes often exhibit a more formal and professional tone. Technically, this approach involves two stages: first training a vectorizer, then training a logistic regression model. This separation can introduce bias, potentially limiting the model’s ability to generalize across datasets with varying characteristics. From this perspective, DistilBERT offers better potential. As an end-to-end model, DistilBERT processes tokenized input and generates task-specific output in a unified manner, which helps mitigate the issues introduced by separate stages of training.

#### Step 3: Fine-tune a DistilBERT model for sentiment analysis. We will follow these steps:

-   **Data Collection**: combine the collected datasets to form a large dataset
-   **Data Preprocessing**: use the tokenizer associated with the pretrained distilbert model to tokenize sentences.
-   **Model Training**: fine-tune the DistilBERT model using the collected film review datasets. To reduce time and computational costs associated with training on the entire dataset,we randomly select 25% of the data for training.
-   **Model Evaluation**: evaluate the model on the testset and compare its performance with that of a simple logistic regression model






**Data Collection** The two datasets have different feature structures. In the first dataset, the label feature is ClassLabel, while in the second dataset, it is Value. To align them, we will manually modify the second feature of the first dataset before concatenating the two datasets.

In [10]:
print(dataset_rt['train'].features)
print(dataset_imdb['train'].features)

{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['neg', 'pos'], id=None)}
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None)}


In [ ]:
from datasets import DatasetDict, Features, Value

# Define the new features to explicitly set label to int64
new_features = Features({
    "text": Value("string"),
    "label": Value("int64")
})

# Convert ClassLabel to int and update schema
dataset_rt_aligned = DatasetDict({
    split: dataset.map(
        lambda example: {"label": int(example["label"])},  # Convert label to int
        features=new_features                             # Update schema
    )
    for split, dataset in dataset_rt.items()
})

# Confirm the change
print(dataset_rt_aligned['train'].features)

In [12]:
# Combine datasets
combined_dataset = {
    "train": concatenate_datasets([dataset_rt_aligned["train"], dataset_imdb["train"]]),
    "test": concatenate_datasets([dataset_rt_aligned["test"], dataset_imdb["test"]]),
    "validation": concatenate_datasets([dataset_rt_aligned["validation"], dataset_imdb["validation"]]),
}

combined_dataset

{'train': Dataset({
     features: ['text', 'label'],
     num_rows: 44530
 }),
 'test': Dataset({
     features: ['text', 'label'],
     num_rows: 11066
 }),
 'validation': Dataset({
     features: ['text', 'label'],
     num_rows: 5066
 })}

**Data Preprocessing** The following steps are the standard procedure of preprocessing text data for a NLP task using the Hugging Face Transformers library. The tokenizer tokenizes and formats the raw text for the DistilBERT model, and the data collator prepares data batches for training or inference.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = {
    "train": combined_dataset["train"].map(preprocess_function, batched=True),
    "test": combined_dataset["test"].map(preprocess_function, batched=True),
    "validation": combined_dataset["validation"].map(preprocess_function, batched=True),
}

train_dataset = tokenized_dataset["train"].shuffle(seed=42)
eval_dataset = tokenized_dataset["test"].shuffle(seed=42)

# we might only want to use a subset of the training data for faster training
total_train_size = len(tokenized_dataset["train"])
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(total_train_size // 4))
small_eval_dataset = tokenized_dataset["validation"].shuffle(seed=42).select(range(len(eval_dataset) // 4))

In [14]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

**Model Training**

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="stats_507_model",
    learning_rate=1e-5,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to = "none",
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

<ipython-input-19-65df68d9c03a>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.364100,0.270784,0.895517
2,0.230900,0.311933,0.898409
3,0.146300,0.373838,0.895517
4,0.110100,0.424302,0.896963
5,0.088500,0.447061,0.897325


TrainOutput(global_step=3480, training_loss=0.1739361686268072, metrics={'train_runtime': 611.5856, 'train_samples_per_second': 91.009, 'train_steps_per_second': 5.69, 'total_flos': 7212256858229904.0, 'train_loss': 0.1739361686268072, 'epoch': 5.0})

Since model training can be time-consuming, it is beneficial to save the model parameters for future use. This allows us to load the model directly without retraining. The code below compresses the `checkpoint-xxx` folder into a ZIP file and downloads it (uncomment before running, adjust the number manually if necessary).

In [ ]:
# !zip -r checkpoint-126.zip stats_507_model/checkpoint-126

In [ ]:
# from google.colab import files
# files.download('checkpoint-126.zip')

Uncomment the code below and run it to unzip the file

In [ ]:
# !unzip checkpoint-126.zip

**Model Evaluation** After fine-tuning the DistilBERT model, print the model's performance on the training and testing datasets. In particular, we want to see if there a difference in model's ability to correctly classify movie reviews from IMDB and rotten tomatoes datasets.

In [20]:
from transformers import pipeline
from evaluate import load
from tqdm import tqdm
import torch

# Verify GPU availability and set the device
device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. Using CPU.")

# Load the fine-tuned DistilBERT model and tokenizer
classifier = pipeline(
    'sentiment-analysis',
    model='stats_507_model/checkpoint-696',
    tokenizer='stats_507_model/checkpoint-696',
    max_length=512,
    truncation=True,
    device=device
)

GPU is available: NVIDIA A100-SXM4-40GB


In [21]:
combined_dataset = {
    "train": {
        "imdb": dataset_imdb["train"],
        "rotten_tomatoes": dataset_rt_aligned["train"]
    },
    "test": {
        "imdb": dataset_imdb["test"],
        "rotten_tomatoes": dataset_rt_aligned["test"]
    },
    "validation": {
        "imdb": dataset_imdb["validation"],
        "rotten_tomatoes": dataset_rt_aligned["validation"]
    }
}

In [22]:
def evaluate_model(dataset, dataset_name):
    print(f"Evaluating on {dataset_name}")

    predictions = []
    references = []

    # Perform prediction on the dataset
    for sample in tqdm(dataset):
        text = sample["text"]
        true_label = sample["label"]

        # Get the predicted label
        result = classifier(text)[0]
        predicted_label = 1 if result['label'] == 'POSITIVE' else 0

        predictions.append(predicted_label)
        references.append(true_label)

    # Generate classification metrics
    report = classification_report(references, predictions, target_names=["NEGATIVE", "POSITIVE"], output_dict=True)

    # Print evaluation results in table format
    print(f"\nClassification Metrics for {dataset_name}:")
    print(f"{'Metric':<15}{'NEGATIVE':<15}{'POSITIVE':<15}")
    print("-" * 45)
    for metric in ["precision", "recall", "f1-score"]:
        print(f"{metric.capitalize():<15}{report['NEGATIVE'][metric]:<15.2f}{report['POSITIVE'][metric]:<15.2f}")
    print(f"{'Accuracy':<15}{accuracy_score(references, predictions):<15.2f}")

In [23]:
print("**Model Evaluation**")
evaluate_model(combined_dataset["train"]["imdb"], "IMDb Train Dataset")
evaluate_model(combined_dataset["train"]["rotten_tomatoes"], "Rotten Tomatoes Train Dataset")
evaluate_model(combined_dataset["test"]["imdb"], "IMDb Test Dataset")
evaluate_model(combined_dataset["test"]["rotten_tomatoes"], "Rotten Tomatoes Test Dataset")

**Model Evaluation**
Evaluating on IMDb Train Dataset


100%|██████████| 36000/36000 [03:29<00:00, 172.02it/s]



Classification Metrics for IMDb Train Dataset:
Metric         NEGATIVE       POSITIVE       
---------------------------------------------
Precision      0.93           0.91           
Recall         0.90           0.94           
F1-score       0.92           0.92           
Accuracy       0.92           
Evaluating on Rotten Tomatoes Train Dataset


100%|██████████| 8530/8530 [00:41<00:00, 206.31it/s]



Classification Metrics for Rotten Tomatoes Train Dataset:
Metric         NEGATIVE       POSITIVE       
---------------------------------------------
Precision      0.83           0.86           
Recall         0.87           0.82           
F1-score       0.85           0.84           
Accuracy       0.85           
Evaluating on IMDb Test Dataset


100%|██████████| 10000/10000 [00:58<00:00, 170.97it/s]



Classification Metrics for IMDb Test Dataset:
Metric         NEGATIVE       POSITIVE       
---------------------------------------------
Precision      0.93           0.90           
Recall         0.90           0.93           
F1-score       0.91           0.91           
Accuracy       0.91           
Evaluating on Rotten Tomatoes Test Dataset


100%|██████████| 1066/1066 [00:05<00:00, 203.31it/s]


Classification Metrics for Rotten Tomatoes Test Dataset:
Metric         NEGATIVE       POSITIVE       
---------------------------------------------
Precision      0.81           0.84           
Recall         0.85           0.80           
F1-score       0.83           0.82           
Accuracy       0.83           


The following code can be used to fine-tune DistilBERT on the entire collected dataset, which is expected to improve the model's performance.

In [ ]:
training_args = TrainingArguments(
    output_dir="stats_507_model/using_full_dataset",
    learning_rate=1e-5,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to = "none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()